# NIDS Exploratory Analysis
Use this notebook for class distribution, feature exploration, correlation analysis, and model evaluation.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data_collection import load_raw_data
from src.preprocessing import clean_data

df = load_raw_data(ROOT / "data" / "raw" / "traffic.csv")
df = clean_data(df)
df.head()

## Class distribution

In [ ]:
counts = df["Label"].value_counts()
print(counts)
counts.plot(kind="bar", title="Label distribution")
plt.ylabel("Count")
plt.show()

## Feature distributions (benign vs. attack)

In [ ]:
from src.feature_extraction import DEFAULT_FEATURES

binary_label = df["Label"].apply(lambda x: "BENIGN" if str(x).upper() == "BENIGN" else "ATTACK")
available = [f for f in DEFAULT_FEATURES if f in df.columns]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, feat in zip(axes.ravel(), available):
    for label, group in df.groupby(binary_label):
        ax.hist(group[feat], bins=30, alpha=0.5, label=label)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Correlation heatmap

In [ ]:
import numpy as np

corr = df[available].corr()
plt.figure(figsize=(9, 7))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(len(available)), available, rotation=90)
plt.yticks(range(len(available)), available)
plt.colorbar()
plt.title("Feature correlation")
plt.tight_layout()
plt.show()

## Load the trained model and evaluate

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from src.predict import load_model
from src.preprocessing import preprocess

bundle = load_model()
model, features = bundle["model"], bundle["features"]

X, y, _ = preprocess(df, target_column="Label")
y_binary = y.apply(lambda x: "BENIGN" if x.upper() == "BENIGN" else "ATTACK")
X = X[features]

preds = model.predict(X)
print(classification_report(y_binary, preds))
ConfusionMatrixDisplay.from_predictions(y_binary, preds)
plt.show()

## Feature importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importances.plot(kind="barh", figsize=(8, 6), title="Feature importance (Random Forest)")
plt.gca().invert_yaxis()
plt.show()